# SFT LoRA — Qwen3.5-2B

Chấm bằng **đúng code của BTC** (`third_party/scoring.py` + `rouge_score/`), chọn
checkpoint theo **METEOR**, không theo loss và không theo trung bình hai metric. Lý do đo
được, trên 300 cặp thật:

| baseline | METEOR | ROUGE-L |
|---|---:|---:|
| chính reference | 1,0000 | 1,0000 |
| nửa đầu reference | 0,4791 | 0,6650 |
| reference xáo từ | 0,5080 | 0,3524 |
| reference của câu KHÁC | 0,1170 | 0,2780 |

ROUGE-L có **sàn 0,278** cho câu sai hẳn (tokenizer của nó xoá dấu, khớp mảnh ASCII) và
**thưởng trả lời cụt** (0,665 cho nửa đầu). Chọn theo trung bình là dạy model nói ngắn.

Loss cũng không dùng làm tiêu chí dừng: ~88% đáp án mở đầu cùng khuôn "Căn cứ Điều N…",
loss vẫn giảm đúng lúc model chuyển sang học thuộc khuôn — đúng dạng lỗi của v9.2.

**Ngân sách.** T4 ~3 s/mẫu (seq p50 3.378, LoRA + gradient checkpointing) → ~4,5 giờ cho
một epoch 5.400 mẫu, cộng ~15 phút mỗi lần eval sinh 120 câu × 768 token. Phiên Kaggle
chết ở 12 giờ và **output không được lưu nếu kernel chết vì hết giờ**, nên
`--gio-toi-da 10.5` dừng chủ động và ghi `cuoi/`. Phiên sau chạy tiếp bằng `--tiep`.

In [ ]:
import os, sys, shutil, zipfile, json
from pathlib import Path

# Chỉ nhận codebase từ dataset `legalqa-v10-src`. Kernel gắn qua `kernel_sources` mang
# theo cả `repo/` của lần chạy trước, và glob bắt trúng bản CŨ đó trước — v10-train-sft
# v2 chết vì `scripts/train_sft.py` không tồn tại trong bản 3ea2413 nằm trong output của
# v10-build-sft. Lọc theo tên dataset thì không có chuyện đó.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
if SRC is None:
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

# Ghim 5.14.1: `model_type` của model nền là `qwen3_5`
# (`Qwen3_5ForConditionalGeneration`), và bản 5.0.0 dựng sẵn của Kaggle KHÔNG biết kiến
# trúc đó — cả ba lớp Auto* đều trả "Transformers does not recognize this architecture".
# 5.14.1 là bản v9.2 đã chạy được với đúng checkpoint này.
!pip -q install transformers==5.14.1 peft nltk
# Gỡ torchao: bản dựng sẵn của Kaggle là 0.10.0, còn peft mới yêu cầu >0.16 và
# `dispatch_torchao` NÉM ImportError ngay giữa `get_peft_model` — không phải bỏ qua,
# là chết. Không lượng tử hoá gì ở đây nên gỡ hẳn rẻ hơn nâng (nâng torchao kéo theo
# torch, mà torch của Kaggle đang khớp driver CUDA của T4).
!pip -q uninstall -y torchao

os.chdir(REPO)
sys.path.insert(0, "src")
os.environ["PYTHONPATH"] = "src"

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())
import transformers
print("transformers:", transformers.__version__)
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (5, 14), \
    "transformers quá cũ, không biết model_type qwen3_5 — cell pip phía trên chưa ăn"
print("GPU     :", torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability()
print("sm      :", f"{cap[0]}.{cap[1]}", "(fp16 OK)" if cap[0] >= 7 else "(fp16 KHÔNG chạy — đổi sang T4)")
# Dataset Kaggle tạo version BẤT ĐỒNG BỘ: đẩy xong mà kernel chạy ngay thì nó vẫn nhận
# bản cũ. Lần trước tốn 4 phút GPU mới lòi ra ở `unrecognized arguments: --nguon`. Kiểm
# ngay ở đây để hỏng trong 2 giây, không phải sau khi tải xong reranker.
_can = ["--nguon", "--gold", "--giu-het", "--ra"] if "build_dev" in "kaggle/train_sft.ipynb" else ["--tiep", "--gio-toi-da"]
_f = "scripts/build_sft.py" if "build_dev" in "kaggle/train_sft.ipynb" else "scripts/train_sft.py"
_s = (REPO / _f).read_text()
_thieu = [c for c in _can if c not in _s]
assert not _thieu, f"codebase cũ, {_f} thiếu {_thieu} — chờ dataset legalqa-v10-src tạo xong version rồi Add Input lại"


## Dữ liệu và model nền

`sft.jsonl` lấy từ output kernel `v10-build-sft`, `dev.jsonl` từ `v10-build-dev`. Cả hai
vào bằng *Add Input → Notebook Output*, không dựng lại — dựng lại tốn 3,5 giờ GPU.

In [ ]:
SFT = next(Path("/kaggle/input").glob("**/sft.jsonl"), None)
DEV = next(Path("/kaggle/input").glob("**/dev.jsonl"), None)
assert SFT and DEV, f"thiếu dữ liệu: sft={SFT} dev={DEV} — gắn output v10-build-sft và v10-build-dev"

# Tìm bằng tokenizer.json + "qwen" trong đường dẫn: đúng luật đã chạy được ở
# build_sft.ipynb. Lần trước lọc theo "ForCausalLM" trong config.json và trượt — model
# nền là biến thể đa phương thức nên `architectures` KHÔNG phải ForCausalLM.
BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"
cfg = json.loads((BASE / "config.json").read_text())
print("base  ", BASE)
print("kiến trúc", cfg.get("architectures"), "| model_type", cfg.get("model_type"))

# Kaggle đổi tên shard khi upload: file thật là `model.safetensors-00001-of-00001.safetensors`
# nhưng `model.safetensors.index.json` trỏ tới tên chuẩn. Sai tên thì loader ném
# EnvironmentError sau khi đã nuốt vài phút. Vá bằng symlink trong thư mục ghi được.
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
thieu = [n for n in can if not (BASE / n).exists()]
if thieu:
    FIX = Path("/kaggle/working/base")
    FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        d = FIX / f.name
        if not d.exists():
            d.symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} shard index cần"
    for ten, that in zip(can, thuc):
        d = FIX / ten
        if not d.exists():
            d.symlink_to(that)
    BASE = FIX
    print("vá tên shard ->", BASE, thieu)

n_sft = sum(1 for _ in open(SFT, encoding="utf-8"))
n_dev = sum(1 for _ in open(DEV, encoding="utf-8"))
print(f"train {n_sft:,}  dev {n_dev:,}")
os.environ["SFT"], os.environ["DEV"], os.environ["BASE"] = str(SFT), str(DEV), str(BASE)

## Train

`--eval-steps 100` với `accum 16` là eval mỗi 1.600 mẫu — đủ dày để thấy đường METEOR
uốn, đủ thưa để phần sinh không nuốt mất giờ GPU. `--kien-nhan 3` dừng sớm khi ba lần
eval liên tiếp không cải thiện METEOR.

Đọc log theo ba số: `loss` giảm đều, `gnorm` **không** vọt (ghi trước khi clip nên nó
nói thật), và `METEOR` ở mỗi mốc. Nếu `meteor_truot` bám sát `meteor` thì model đang bịa
tự tin khi context thiếu căn cứ — tập train đã loại hết nhóm đó nên nó chưa từng thấy.
`--max-new 1024` chứ không 768: đo trên 500 answer của dev, 768 token cắt cụt 8,4% mẫu
còn 1024 chỉ cắt 2,2% (p50 398, p90 730, p95 885). Sinh dừng ở EOS nên trần cao hơn gần
như không tốn thêm giờ — cái tốn giờ là độ dài THẬT, không phải trần. Cắt cụt làm METEOR
tụt đều ở mọi checkpoint nên không đổi thứ tự chọn, nhưng nó làm con số dev lệch khỏi
leaderboard, mà đó là con số sẽ được đem đi so.


In [ ]:
# `!python ...` KHÔNG ném theo mã thoát: v6 chết ở TypeError trong ma_hoa nhưng Kaggle
# vẫn báo COMPLETE, và Monitor tin trạng thái đó. `subprocess.run(check=True)` mới làm
# notebook đỏ khi train chết. (Bỏ `| tail` ở v6 chỉ chữa được một nửa nguyên nhân.)
import subprocess, sys
subprocess.run([sys.executable, "scripts/train_sft.py",
    "--model", str(BASE), "--data", str(SFT), "--dev", str(DEV),
    "--ra", "/kaggle/working/out",
    "--epochs", "1.0", "--lr", "1e-4", "--r", "32", "--alpha", "64",
    "--accum", "16", "--eval-steps", "100", "--dev-n", "120",
    "--max-new", "1024", "--gen-bs", "4", "--gio-toi-da", "10.5",
], check=True)


In [ ]:
p = Path("/kaggle/working/out/lich_su.json")
if not p.exists():
    print("chưa có lich_su.json — train chết trước lần eval đầu, đọc traceback ở cell trên")
else:
    h = json.loads(p.read_text())
    print(f"{'bước':>6} {'loss':>7} {'gnorm':>6} {'METEOR':>7} {'ROUGE':>7} {'M-trượt':>8} {'dài':>6}")
    for d in h:
        print(f"{d['buoc']:6} {d['loss']:7.4f} {d['gnorm']:6.2f} {d['meteor']:7.4f} "
              f"{d['rouge']:7.4f} {d.get('meteor_truot', float('nan')):8.4f} {d['do_dai_tb']:6.0f}")